# Hospital Excess Readmission Classification — Data Cleaning and Structural Validation

This notebook loads the CMS Hospital Readmissions Reduction Program dataset and prepares it for later classification modeling. The purpose of this notebook is to understand the dataset structure, identify missingness patterns, and validate the grain of the data before creating a predictive target.

This project uses hospital-level public CMS HRRP data. Each row is expected to represent a hospital-measure record, not an individual patient readmission.

## Load and Standardize Dataset

The dataset is loaded from the local project data folder. Column names are standardized to lowercase with underscores to make the dataset easier to work with in Python.

## Initial Data Inspection

Before cleaning, I inspected the dataset shape, column names, data types, missing values, and duplicate rows. This helps identify any structural issues that need to be cleaned before making any changes to the data.

## Missing Value Investigation

Several important analytical columns contain missing values, including excess readmission ratio, predicted readmission rate, expected readmission rate, and number of readmissions. Because these columns are central to the project, missing values should not be dropped blindly. Instead, I investigate whether the missingness is related to CMS reporting rules.

## Missing ERR by Measure and Footnote

Rows with missing excess readmission ratio are reviewed by measure name and CMS footnote. This helps determine whether the missing values are random data quality problems or intentional reporting exclusions.

## Create Filtered Analytical Dataset

Rows with missing excess readmission ratio cannot be used for supervised classification because the target variable cannot be constructed. These rows are excluded from the modeling dataset but should still be discussed as structurally excluded records.

## Validate Dataset Grain

Before modeling, I validate the dataset grain by checking whether the combination of facility_id and measure_name uniquely identifies each row. This confirms whether each row represents a unique hospital-measure observation.

## Hospital Representation After Filtering

After filtering to valid ERR rows, hospitals may have varying numbers of measure records. This is expected because some measures have more structural missingness than others. The filtered dataset represents reportable hospital-measure records, not all hospitals equally.

In [ ]:
#Import path for OS-independent file paths
from pathlib import Path

#Import pandas for data loading and analysis
import pandas as pd

#Relative Path to Dataset for safety
csv_path = Path("../data") / "FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv"
df_raw = pd.read_csv(csv_path)

#Standardize column names - convert to lowercase and replace spaces with underscores
df_raw.columns = ( df_raw.columns.
    str.lower(). 
    str.replace(" ", "_")
)

#Inital Inspection of Dataset Structure
df_raw.shape
df_raw.head()
df_raw.info()
df_raw.columns

#Check the number of missing values for each column
df_raw.isna().sum()

#Check for any duplicated rows (0)
df_raw.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 18330 entries, 0 to 18329
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   facility_name               18330 non-null  str    
 1   facility_id                 18330 non-null  int64  
 2   state                       18330 non-null  str    
 3   measure_name                18330 non-null  str    
 4   number_of_discharges        8242 non-null   float64
 5   footnote                    6987 non-null   float64
 6   excess_readmission_ratio    11720 non-null  float64
 7   predicted_readmission_rate  11720 non-null  float64
 8   expected_readmission_rate   11720 non-null  float64
 9   number_of_readmissions      11720 non-null  str    
 10  start_date                  18330 non-null  str    
 11  end_date                    18330 non-null  str    
dtypes: float64(5), int64(1), str(6)
memory usage: 3.0 MB


np.int64(0)

In [2]:
#Find the correlation between measure categories and missing ERR values
df_raw[df_raw['excess_readmission_ratio'].isna()]['measure_name'].value_counts()

#Find whether discharge counts exist for rows that are missing ERR
df_raw[df_raw['excess_readmission_ratio'].isna()]['number_of_discharges'].value_counts()

#Investigate if there is an association between footnotes and missing ERR values
df_raw[df_raw['excess_readmission_ratio'].isna()]['footnote'].value_counts()


footnote
5.0    3255
1.0    3150
7.0     205
Name: count, dtype: int64

In [3]:
#Create a filtered analytical dataset containing rows that have valid
# excess readmission ratio values
df_valid_err = df_raw.copy()
df_valid_err = df_valid_err.dropna(subset = ['excess_readmission_ratio'])

#Confirm filtered dataset shape
df_valid_err.shape

# Compare measure distribution after filtering valid ERR rows
df_valid_err['measure_name'].value_counts()

# Compare against original measure distribution before filtering
df_raw['measure_name'].value_counts()

# Check which footnotes remain after filtering
df_valid_err['footnote'].value_counts()

# Compare average discharge counts before and after filtering
df_raw['number_of_discharges'].mean()
df_valid_err['number_of_discharges'].mean()

# Preliminary class balance inspection:
# Count rows where ERR indicates excess readmissions
(df_valid_err['excess_readmission_ratio'] > 1).sum()

# Same check on original dataset
# (NaN values evaluate False during comparison)
(df_raw['excess_readmission_ratio'] > 1).sum()

# Count rows where ERR is less than or equal to threshold
(df_valid_err['excess_readmission_ratio'] <= 1).sum()

# Same threshold check on original dataset
(df_raw['excess_readmission_ratio'] <= 1).sum()



np.int64(6077)

In [4]:
#Confirm filtered dataset shape
df_valid_err.shape
#(11720, 12)

# Check whether facility_id + measure_name uniquely identiifes rows
df_valid_err[['facility_id', 'measure_name']].drop_duplicates().shape[0]
#11720 

# Save processed dataset for target analysis and modeling notebooks
df_valid_err.to_parquet("../data/processed/valid_err.parquet")
